In [1]:
import json
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
import os
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv()
api_key = os.getenv("OpenAi_API_key")
openai = OpenAI(api_key=api_key)
# Create Chroma client
client = chromadb.PersistentClient(path="./chroma_db")

In [3]:
# OpenAI embedding function
embedding_function = OpenAIEmbeddingFunction(
    api_key=api_key,
    model_name="text-embedding-3-small"
)

In [4]:
# Create collection
collection = client.get_or_create_collection(
    name="netflix_titles",
    embedding_function=embedding_function
)


In [5]:
# Load data
with open("netflix_titles.json", "r", encoding="utf-8") as f:
    data = json.load(f)

documents = []
metadatas = []
ids = []

for item in data:
    document = f"""
    Title: {item['title']}
    Type: {item['type']}
    Genres: {item['listed_in']}
    Description: {item['description']}
    """

    documents.append(document)

    metadatas.append({
        "title": item["title"],
        "type": item["type"],
        "release_year": item["release_year"]
    })

    ids.append(item["show_id"])

    #print(f"Adding{len(documents)} documents to the collection")

In [6]:
# Insert data
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

In [7]:
print(f"Inserted {len(ids)} records")

Inserted 5 records


In [11]:
result = collection.query(
    query_texts=["films about dogs"],
    n_results=3
)
#print(results)

In [12]:
print("\nTop Matches:\n")

for i in range(len(result["ids"][0])):
    print("=" * 50)
    print("ID:", result["ids"][0][i])
    print("Metadata:", result["metadatas"][0][i])
    print("Document:", result["documents"][0][i])
    print("Distance:", result["distances"][0][i])


Top Matches:

ID: s1
Metadata: {'release_year': 2018, 'type': 'Movie', 'title': 'Dog Days'}
Document: 
    Title: Dog Days
    Type: Movie
    Genres: Comedy, Drama
    Description: A group of people discover how their lives become connected through their lovable dogs.
    
Distance: 0.3917437195777893
ID: s2
Metadata: {'title': 'Benji', 'type': 'Movie', 'release_year': 2018}
Document: 
    Title: Benji
    Type: Movie
    Genres: Family Movies
    Description: A determined dog helps rescue two kidnapped children.
    
Distance: 0.5087330341339111
ID: s3
Metadata: {'release_year': 2016, 'title': 'The Secret Life of Pets', 'type': 'Movie'}
Document: 
    Title: The Secret Life of Pets
    Type: Movie
    Genres: Animation, Family
    Description: A terrier and his canine friends embark on an adventure through New York City.
    
Distance: 0.5133501291275024
